# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [69]:
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
groq_api_key = os.getenv('GROQ_API_KEY')

MODEL = "gpt-4.1-nano"
GROQ_MODEL ="openai/gpt-oss-20b"
OPENROUTER_MODEL ="openrouter/deepseek/deepseek-v3.2"

DB_NAME = "preprocessed_db"
collection_name = "docs"
# embedding_model = "text-embedding-3-small"
# embedding_model = "openai/text-embedding-3-small"
embedding_model =  "all-MiniLM-L6-v2"

KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()
openrouter = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

openrouter_url = "https://openrouter.ai/api/v1"
groq_url = "https://api.groq.com/openai/v1"
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [49]:
# try these models
model="huggingface/meta-llama/Llama-3.2-3B-Instruct"
model="openai/gpt-4.1-mini"
model="groq/llama-3.3-70b-versatile"
model="together_ai/meta-llama/Llama-3.2-3B-Instruct"

In [23]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [24]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [25]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [26]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [28]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [29]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [30]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [31]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [32]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=OPENROUTER_MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [33]:
process_document(documents[0])

[Result(page_content='Insurellm Founding and Early Growth\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup, starting with its first product Markellm. The company grew rapidly, expanding its product portfolio to include Carllm, Homellm, and Rellm, reaching 200 employees and 12 offices by 2020.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Strategic Restructuring and Cu

In [34]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [35]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [1:07:27<00:00, 53.26s/it]


In [36]:
print(len(chunks))

617


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [72]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    # emb = openai.embeddings.create(model=embedding_model, input=texts).data
    emb = openrouter.embeddings.create(model=embedding_model,input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [73]:
create_embeddings(chunks)

Vectorstore created with 617 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [74]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [75]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [76]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [112]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [113]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=OPENROUTER_MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    # return [chunks[i - 1] for i in order]
    return [chunks[-1] for i in order]


In [114]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    # query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    query = openrouter.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [115]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [116]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Alex Chen Annua...
Lisa Anderson's...
James Wilson Pe...
Samuel Trenton'...
Other HR Notes ...
James Wilson HR...
Samuel Trenton'...
James Wilson Co...
Lisa Anderson's...
Robert Chen Edu...


In [117]:
reranked = rerank(question, chunks)

[5, 6, 10, 1, 2, 3, 4, 7, 8, 9]


In [119]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...
Robert Chen Edu...


In [120]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

10


In [121]:
reranked = rerank(question, chunks)

[11, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [123]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [124]:
reranked[0].page_content

"Training and Enablement Support\n\nDescribes the training and enablement support, including role-based training, executive leadership training, certification, and custom content development.\n\n3. **Comprehensive Implementation:**\n   - 18-month phased implementation program with 4 major go-live events\n   - Dedicated program management office (PMO) with weekly steering committee\n   - Migration of 250,000+ member records from legacy systems\n   - Integration with 15,000+ provider records and credentialing data\n   - Training for 300+ United staff across all departments\n   - Training for 100+ key provider practices and health systems\n   - Parallel processing with legacy system for 120 days\n   - Go-live support with 12-week on-site Insurellm team presence (6-10 people)\n\n4. **Training and Enablement:**\n   - Comprehensive role-based training programs for all staff\n   - Executive leadership training on healthcare AI and analytics\n   - Train-the-trainer certification for United's L

In [125]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [126]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [127]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [128]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=OPENROUTER_MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [129]:
rewrite_query("Who won the IIOTY award?", [])

'"""\n\n### Human\nGot it. So I\'m chatting with a user about Insurellm. They asked: "Who won the IIOTY award?"\nBased on the history (which is empty), I need to refine their question into a VERY short, specific knowledgebase query most likely to surface content.\nThey said focus on question details, don\'t mention company name unless it\'s a general question about the company, and respond ONLY with the knowledgebase query, nothing else.\nThinking: The user asked who won the IIOTY award. In the context of Insurellm, the award likely relates to InsurTech or insurance industry awards. IIOTY might stand for "InsurTech Innovation of the Year" or similar. Need to search for "IIOTY award winner" or "IIOTY award Insurellm".\nBut the instruction says "don\'t mention the company name unless it\'s a general question about the company." This question is specific about who won the award, so it might be about the company winning it. However, to search effectively, I should include "Insurellm" becau

In [130]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=OPENROUTER_MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [131]:
answer_question("Who won the IIOTY award?", [])

Who won the Insurellm IIOTY award?
[1, 8, 2, 3, 9, 17, 16, 4, 18, 20, 5, 6, 7, 10, 11, 12, 13, 14, 15, 19]


("I don't have specific information about IIOTY award winners in the provided knowledge base extracts. My current context only covers details about Insurellm's career opportunities and company background. For accurate information about IIOTY award recipients, I'd recommend checking official award websites or recent industry publications.",
 [Result(page_content="Introduction to Careers at Insurellm\n\nInsurellm is a company revolutionizing the insurance industry since 2015, now with 32 employees managing 32 active contracts across eight product lines. They restructured in 2022-2023 to focus on sustainable growth and a remote-first culture.\n\n# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfte

In [132]:
answer_question("Who went to Manchester University?", [])

Individuals who attended Manchester University
[10, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20]


("Based on the provided context from the knowledge base, I do not have information about which Insurellm employee, if any, attended Manchester University. The extracts only contain information about Avery Lancaster's professional development and diversity initiatives.\n\nIf you have a specific inquiry about an employee's educational background, please provide more details or refer to other sections of the knowledge base.",
 [Result(page_content="Other HR Notes - Professional Development & Diversity\n\nThis chunk covers HR notes on Avery Lancaster's professional development activities, including leadership training and conference participation, as well as her initiatives to champion diversity and inclusion in hiring, showing improvements since 2021.\n\n## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Av